# Robust Bank Account Fraud Detection Under Temporal Distribution Shift
**BT4012 Fraud Analytics — Group 21**

**Primary question:** How do temporal changes in fraud prevalence and predictive feature relationships affect the future performance of bank account fraud detection models?

**Supporting questions**
1. Which classifier best maintains fraud-detection performance on future applications?
2. How does performance differ under changes in fraud prevalence (Variant IV) compared with changes in feature predictiveness (Variant V)?
3. How do these changes affect false-positive rates and fraud recall for applicants aged ≤50 vs >50?

**Design summary**

| Item | Choice |
|---|---|
| Datasets | Base, Variant IV, Variant V (BAF suite, NeurIPS 2022), each a separate experiment |
| Time split | Train: months 0–3 · Validation: months 4–5 · Out-of-time test: months 6–7 (untouched until evaluation) |
| Models | Logistic regression, decision tree, random forest, LightGBM, XGBoost (class-weighted) |
| Tuning | Optuna (seeded TPE) on the Base validation period, then fixed for Variants IV and V |
| Ranking metrics | Average precision (AP) **and** recall at 5% FPR (prevalence-independent) |
| Operating settings | **Fixed threshold** (5% FPR on validation, applied unchanged) — primary for the age comparison; **fixed review capacity** (top 1% / 5% of each month) — secondary |

**Data:** download `Base.csv`, `Variant IV.csv` and `Variant V.csv` from the [Kaggle BAF dataset](https://www.kaggle.com/datasets/sgpjesus/bank-account-fraud-dataset-neurips-2022) into the folder set in `DATA_DIR` below.

## 0. Setup

In [ ]:
import os
import time
import json
import warnings
from itertools import product, combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import lightgbm as lgb
import xgboost as xgb
import optuna

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, precision_recall_curve, roc_auc_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.4f}".format)
optuna.logging.set_verbosity(optuna.logging.WARNING)

# results (random forest especially) shift slightly between library versions, so log them
print("numpy", np.__version__, "| pandas", pd.__version__, "| sklearn", sklearn.__version__,
      "| lightgbm", lgb.__version__, "| xgboost", xgb.__version__, "| optuna", optuna.__version__)

In [ ]:
DATA_DIR = "data"
DATASET_FILES = {"Base": "Base.csv", "Variant IV": "Variant IV.csv", "Variant V": "Variant V.csv"}
OUT_DIR = "outputs"
os.makedirs(OUT_DIR, exist_ok=True)

TARGET = "fraud_bool"
MONTH = "month"
AGE = "customer_age"
AGE_CUTOFF = 50
INCLUDE_AGE_FEATURE = True

TRAIN_MONTHS = [0, 1, 2, 3]
VAL_MONTHS = [4, 5]
TEST_MONTHS = [6, 7]

TARGET_FPR = 0.05                  # operating point used in the BAF paper
REVIEW_CAPACITIES = [0.01, 0.05]   # share of each month's applications sent for review

SAMPLE_FRAC = None   # e.g. 0.2 for a quick run
RANDOM_STATE = 42
N_THREADS = 4        # boosting results depend on thread count, so pin it

# Optuna trials per model. the boosting models get more because they have more knobs
N_TRIALS = {"Logistic Regression": 20, "Decision Tree": 30, "Random Forest": 20,
            "LightGBM": 40, "XGBoost": 40}

CAT_COLS = ["payment_type", "employment_status", "housing_status", "source", "device_os"]
# BAF codes missing values as -1 in these columns...
MISSING_NEG1 = ["prev_address_months_count", "current_address_months_count", "bank_months_count",
                "session_length_in_minutes", "device_distinct_emails_8w"]
# ...and as any negative value in this one
MISSING_NEGATIVE = ["intended_balcon_amount"]

## 1. Load data
Missing-value sentinels are converted to `NaN` so they can be imputed and flagged. Numeric columns are downcast to save memory (3 × 1M rows).

In [ ]:
datasets = {}
for name, file in DATASET_FILES.items():
    df = pd.read_csv(os.path.join(DATA_DIR, file))

    if SAMPLE_FRAC:
        # stratify so each month keeps its fraud rate
        df = df.groupby([MONTH, TARGET], group_keys=False).sample(frac=SAMPLE_FRAC, random_state=RANDOM_STATE)
        df = df.sort_index()

    # 3 x 1M rows: downcast to keep memory manageable
    for col in df.columns:
        if df[col].dtype == "float64":
            df[col] = df[col].astype("float32")
        elif df[col].dtype == "int64":
            df[col] = pd.to_numeric(df[col], downcast="integer")

    for col in MISSING_NEG1:
        df[col] = df[col].replace(-1, np.nan).astype("float32")
    for col in MISSING_NEGATIVE:
        df[col] = df[col].where(df[col] >= 0).astype("float32")

    df["age_group"] = np.where(df[AGE] > AGE_CUTOFF, ">50", "<=50")

    df["split"] = "test"
    df.loc[df[MONTH].isin(TRAIN_MONTHS), "split"] = "train"
    df.loc[df[MONTH].isin(VAL_MONTHS), "split"] = "val"

    df = df.reset_index(drop=True)
    datasets[name] = df
    print(name, "| rows:", len(df), "| cols:", df.shape[1], "| fraud rate:", round(df[TARGET].mean() * 100, 3), "%")

## 2. Exploratory analysis
### 2.1 Overview and class imbalance

In [ ]:
overview_rows = []
for name, df in datasets.items():
    overview_rows.append({
        "dataset": name,
        "rows": len(df),
        "frauds": df[TARGET].sum(),
        "fraud_rate": df[TARGET].mean(),
        "share_age_>50": (df["age_group"] == ">50").mean(),
        "train_rows": (df["split"] == "train").sum(),
        "val_rows": (df["split"] == "val").sum(),
        "test_rows": (df["split"] == "test").sum(),
    })
overview = pd.DataFrame(overview_rows).set_index("dataset")
overview

In [ ]:
missing_share = pd.DataFrame()
for name, df in datasets.items():
    missing_share[name] = df[MISSING_NEG1 + MISSING_NEGATIVE].isna().mean()
print("Share of missing values")
display(missing_share)

for name, df in datasets.items():
    constant_cols = [col for col in df.columns if df[col].nunique(dropna=False) <= 1]
    print(name, "constant columns:", constant_cols)

### 2.2 Monthly fraud prevalence, overall and by age group
This is where Variant IV should differ: group-specific prevalence in months 0–5, similar prevalence in months 6–7.

In [ ]:
monthly_fraud = {}
for name, df in datasets.items():
    rates = df.groupby([MONTH, "age_group"])[TARGET].mean().unstack()
    rates["overall"] = df.groupby(MONTH)[TARGET].mean()
    monthly_fraud[name] = rates
display(pd.concat(monthly_fraud, axis=1).round(4))

fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
for ax, name in zip(axes, datasets):
    rates = monthly_fraud[name]
    ax.plot(rates.index, rates["<=50"], "-o", label="<=50")
    ax.plot(rates.index, rates[">50"], "-s", label=">50")
    ax.plot(rates.index, rates["overall"], "--", label="overall")
    ax.axvspan(TEST_MONTHS[0] - 0.5, TEST_MONTHS[-1] + 0.5, color="grey", alpha=0.15, label="test period")
    ax.set_title(name)
    ax.set_xlabel("month")
    ax.set_ylabel("fraud rate")
axes[0].legend()
plt.tight_layout()
fig.savefig(OUT_DIR + "/fig1_prevalence.png", dpi=200, bbox_inches="tight")
plt.show()

### 2.3 Feature distributions (Base, fraud vs non-fraud)

In [ ]:
base = datasets["Base"]

# binary columns make unreadable histograms, so only plot ones with more than 2 values
numeric_cols = []
for col in base.columns:
    if col not in CAT_COLS + [TARGET, MONTH, "age_group", "split"] and base[col].nunique() > 2:
        numeric_cols.append(col)

fig, axes = plt.subplots(3, 4, figsize=(18, 10))
for ax, col in zip(axes.ravel(), numeric_cols[:12]):
    ax.hist(base[base[TARGET] == 0][col].dropna(), bins=40, density=True, alpha=0.5, label="legit")
    ax.hist(base[base[TARGET] == 1][col].dropna(), bins=40, density=True, alpha=0.5, label="fraud")
    ax.set_title(col, fontsize=9)
axes[0, 0].legend()
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, len(CAT_COLS), figsize=(20, 3.5))
for ax, col in zip(axes, CAT_COLS):
    base.groupby(col)[TARGET].mean().sort_values().plot.barh(ax=ax)
    ax.set_title("fraud rate by " + col)
    ax.set_xlabel("")
plt.tight_layout()
plt.show()

## 3. Preprocessing and models
* Features: all columns except the label, month and helper columns; columns constant in the **training** months are dropped.
* Numerical: median imputation + missing-indicator columns; standard scaling for logistic regression only.
* Categorical: one-hot encoding (unseen categories ignored).
* All preprocessing is fitted on training months only, inside an sklearn `Pipeline`.
* Class imbalance: class weighting (`balanced` / `scale_pos_weight`); validation and test data keep their natural distribution.

In [ ]:
MODEL_NAMES = ["Logistic Regression", "Decision Tree", "Random Forest", "LightGBM", "XGBoost"]


def train_model(train_df, model_name, params):
    features = [c for c in train_df.columns if c not in [TARGET, MONTH, "age_group", "split"]]
    if not INCLUDE_AGE_FEATURE:
        features = [c for c in features if c != AGE]
    # device_fraud_count is constant in training, which would just add a dead column
    features = [c for c in features if train_df[c].nunique(dropna=False) > 1]
    cat_cols = [c for c in features if c in CAT_COLS]
    num_cols = [c for c in features if c not in CAT_COLS]

    y = train_df[TARGET]
    pos_weight = (y == 0).sum() / (y == 1).sum()

    if model_name == "Logistic Regression":
        clf = LogisticRegression(class_weight="balanced", max_iter=2000, **params)
    elif model_name == "Decision Tree":
        clf = DecisionTreeClassifier(class_weight="balanced", random_state=RANDOM_STATE, **params)
    elif model_name == "Random Forest":
        clf = RandomForestClassifier(class_weight="balanced_subsample", n_jobs=-1,
                                     random_state=RANDOM_STATE, **params)
    elif model_name == "LightGBM":
        # multithreaded LightGBM isn't reproducible without deterministic + force_row_wise
        clf = lgb.LGBMClassifier(scale_pos_weight=pos_weight, random_state=RANDOM_STATE, n_jobs=N_THREADS,
                                 deterministic=True, force_row_wise=True, verbose=-1, **params)
    elif model_name == "XGBoost":
        clf = xgb.XGBClassifier(scale_pos_weight=pos_weight, tree_method="hist", eval_metric="aucpr",
                                random_state=RANDOM_STATE, n_jobs=N_THREADS, **params)

    # missingness is itself a fraud signal in BAF, so keep an indicator column after imputing.
    # only logistic regression needs scaling.
    if model_name == "Logistic Regression":
        num_prep = Pipeline([("impute", SimpleImputer(strategy="median", add_indicator=True)),
                             ("scale", StandardScaler())])
    else:
        num_prep = SimpleImputer(strategy="median", add_indicator=True)

    preprocessor = ColumnTransformer([
        ("num", num_prep, num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
    ])
    pipe = Pipeline([("prep", preprocessor), ("clf", clf)])
    pipe.fit(train_df[features], y)
    return pipe, features

## 4. Hyperparameter selection (Base development period only)
Hyperparameters are tuned with Optuna (TPE sampler, seeded) on Base only: each trial trains on months 0–3 and is scored by validation AP on months 4–5. The best setting per model is then **fixed** for all three datasets, so Variants IV and V act as genuine stress tests. The search ranges extend beyond the original small grid, several of whose best values sat at the edge of the grid.

In [ ]:
def suggest_params(trial, model_name):
    if model_name == "Logistic Regression":
        return {"C": trial.suggest_float("C", 1e-4, 10, log=True)}
    if model_name == "Decision Tree":
        return {"max_depth": trial.suggest_int("max_depth", 4, 20),
                "min_samples_leaf": trial.suggest_int("min_samples_leaf", 50, 3000, log=True)}
    if model_name == "Random Forest":
        # trees and row sample are capped so a trial on 550k rows stays around a minute
        return {"n_estimators": 200,
                "max_depth": trial.suggest_categorical("max_depth", [8, 12, 16, 24, None]),
                "min_samples_leaf": trial.suggest_int("min_samples_leaf", 10, 500, log=True),
                "max_samples": trial.suggest_float("max_samples", 0.1, 0.5)}
    if model_name == "LightGBM":
        return {"n_estimators": trial.suggest_int("n_estimators", 200, 1000, step=100),
                "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
                "num_leaves": trial.suggest_int("num_leaves", 8, 128, log=True),
                "min_child_samples": trial.suggest_int("min_child_samples", 20, 500, log=True),
                "subsample": trial.suggest_float("subsample", 0.5, 1.0),
                "subsample_freq": 1,   # LightGBM ignores subsample unless this is set
                "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
                "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10, log=True)}
    if model_name == "XGBoost":
        return {"n_estimators": trial.suggest_int("n_estimators", 200, 1000, step=100),
                "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
                "max_depth": trial.suggest_int("max_depth", 2, 8),
                "min_child_weight": trial.suggest_float("min_child_weight", 1, 50, log=True),
                "subsample": trial.suggest_float("subsample", 0.5, 1.0),
                "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
                "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10, log=True)}


base_train = datasets["Base"][datasets["Base"]["split"] == "train"]
base_val = datasets["Base"][datasets["Base"]["split"] == "val"]

tuning_rows = []
best_params = {}
for model_name in MODEL_NAMES:
    def objective(trial):
        params = suggest_params(trial, model_name)
        start = time.time()
        pipe, features = train_model(base_train, model_name, params)
        ap = average_precision_score(base_val[TARGET], pipe.predict_proba(base_val[features])[:, 1])
        tuning_rows.append({"model": model_name, "trial": trial.number, "params": json.dumps(params),
                            "val_AP": ap, "secs": time.time() - start})
        # fixed values like subsample_freq aren't in trial.params, so keep the full dict
        trial.set_user_attr("params", params)
        return ap

    # seeded sampler + deterministic models = the same search on every run
    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
    study.optimize(objective, n_trials=N_TRIALS[model_name])
    best_params[model_name] = study.best_trial.user_attrs["params"]
    print(model_name, "-> best val AP =", round(study.best_value, 4), best_params[model_name])

tuning = pd.DataFrame(tuning_rows)
tuning.to_csv(OUT_DIR + "/tuning_results.csv", index=False)

## 5. Train on each dataset and score validation + test months
For each dataset, every model is trained on months 0–3 with the fixed hyperparameters and scores months 4–7. The validation scores (months 4–5) set the operating threshold; the test scores (months 6–7) are used only for evaluation.

In [ ]:
pred_frames = []
fitted = {}
for dataset, df in datasets.items():
    train = df[df["split"] == "train"]
    eval_df = df[df["split"] != "train"]
    for model_name in MODEL_NAMES:
        pipe, features = train_model(train, model_name, best_params[model_name])
        fitted[(dataset, model_name)] = (pipe, features)
        pred_frames.append(pd.DataFrame({
            "dataset": dataset,
            "model": model_name,
            "row": eval_df.index,
            "month": eval_df[MONTH].values,
            "split": eval_df["split"].values,
            "y": eval_df[TARGET].values,
            "score": pipe.predict_proba(eval_df[features])[:, 1],
            "age_group": eval_df["age_group"].values,
        }))
        print(dataset, "-", model_name, "done")

preds = pd.concat(pred_frames, ignore_index=True)

# the decision tree produces large blocks of identical scores, so a cut-off can land inside a tie
# and flag all or none of it. tiny noise breaks the ties; AP and PR curves still use the raw score.
rng = np.random.default_rng(RANDOM_STATE)
preds["score_tb"] = preds["score"] + rng.uniform(0, 1e-9, len(preds))
preds.head()

## 6. Evaluation helpers
* **AP** depends on fraud prevalence, so a fall in AP can come from a prevalence change alone.
* **Recall at 5% FPR (self-calibrated)** uses the threshold that gives 5% FPR *on the evaluated set itself*. TPR is computed only among frauds and FPR only among legitimate applications, so this metric does not depend on prevalence and measures pure separability.
* **Fixed threshold:** the threshold that gives 5% FPR on validation is applied unchanged; the share of applications flagged can drift.
* **Fixed capacity:** the top *k*% of each month's scores are flagged (no labels needed); the workload is constant and the threshold moves.

In [ ]:
def recall_at_fpr(y, scores, fpr=TARGET_FPR):
    # threshold is set on legit applications only, so the result doesn't depend on fraud prevalence
    threshold = np.quantile(scores[y == 0], 1 - fpr)
    return (scores[y == 1] > threshold).mean()


def flag_stats(preds_df, flag_col, group_cols):
    outcomes = preds_df[group_cols].copy()
    outcomes["tp"] = preds_df[flag_col] & (preds_df["y"] == 1)
    outcomes["fp"] = preds_df[flag_col] & (preds_df["y"] == 0)
    outcomes["pos"] = preds_df["y"] == 1
    outcomes["neg"] = preds_df["y"] == 0
    outcomes["flag"] = preds_df[flag_col]
    totals = outcomes.groupby(group_cols).sum()
    counts = outcomes.groupby(group_cols).size()

    stats = pd.DataFrame()
    stats["recall"] = totals["tp"] / totals["pos"]
    stats["FPR"] = totals["fp"] / totals["neg"]
    stats["precision"] = totals["tp"] / totals["flag"]
    stats["alert_rate"] = totals["flag"] / counts
    stats["n"] = counts
    stats["frauds"] = totals["pos"]
    return stats.reset_index()

## 7. Ranking performance: AP vs recall at 5% FPR
If AP falls but recall at 5% FPR is stable, the drop is mostly a prevalence effect. If both fall, the model has genuinely lost its ability to separate fraud from legitimate applications.

In [ ]:
def rank_scores(group):
    return pd.Series({
        "prevalence": group["y"].mean(),
        "AP": average_precision_score(group["y"], group["score"]),
        "recall@5%FPR": recall_at_fpr(group["y"].values, group["score_tb"].values),
    })


rank_period = preds.groupby(["dataset", "model", "split"]).apply(rank_scores).reset_index()
rank_month = preds.groupby(["dataset", "model", "month"]).apply(rank_scores).reset_index()

robust = rank_period.pivot_table(index=["dataset", "model"], columns="split",
                                 values=["AP", "recall@5%FPR", "prevalence"])
robust.columns = [metric + "_" + split for metric, split in robust.columns]
robust["AP_change"] = robust["AP_test"] - robust["AP_val"]
robust["recall@5%FPR_change"] = robust["recall@5%FPR_test"] - robust["recall@5%FPR_val"]
robust = robust[["prevalence_val", "prevalence_test", "AP_val", "AP_test", "AP_change",
                 "recall@5%FPR_val", "recall@5%FPR_test", "recall@5%FPR_change"]]
display(robust)

robust.to_csv(OUT_DIR + "/ranking_val_vs_test.csv")
rank_month.to_csv(OUT_DIR + "/ranking_month.csv", index=False)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 8), sharex=True)
for j, dataset in enumerate(datasets):
    for i, metric in enumerate(["AP", "recall@5%FPR"]):
        ax = axes[i, j]
        for model_name in MODEL_NAMES:
            model_month = rank_month[(rank_month["dataset"] == dataset) & (rank_month["model"] == model_name)]
            ax.plot(model_month["month"], model_month[metric], "-o", label=model_name)
        ax.axvspan(TEST_MONTHS[0] - 0.5, TEST_MONTHS[-1] + 0.5, color="grey", alpha=0.15)
        ax.set_title(dataset + " - " + metric)
        ax.set_ylabel(metric)
        if i == 1:
            ax.set_xlabel("month")
axes[0, 0].legend(fontsize=8)
plt.suptitle("Monthly ranking performance (shaded = test months)")
plt.tight_layout()
fig.savefig(OUT_DIR + "/fig2_monthly_ranking.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# Base has no injected shift, so its change is the baseline drift to subtract
drop_vs_base = robust[["AP_change", "recall@5%FPR_change"]].unstack("dataset")
for variant in ["Variant IV", "Variant V"]:
    for metric in ["AP_change", "recall@5%FPR_change"]:
        drop_vs_base[(metric, variant + " - Base")] = drop_vs_base[(metric, variant)] - drop_vs_base[(metric, "Base")]
drop_vs_base.sort_index(axis=1)

### 7.1 Two definitions of robustness (RQ1)
* **Out-of-time performance:** test recall at 5% FPR — what the bank actually gets.
* **Stability:** the change from validation to test, net of the same model's change on Base (`shift_effect`), so normal month-to-month drift is removed and only the injected shift remains. `retention` = test / validation recall.

A model can rank high on one and low on the other (e.g. a weak model has little to lose).

In [ ]:
robustness = robust[["recall@5%FPR_val", "recall@5%FPR_test", "recall@5%FPR_change"]].reset_index()

base_change = robustness[robustness["dataset"] == "Base"].set_index("model")["recall@5%FPR_change"]
robustness["shift_effect"] = robustness["recall@5%FPR_change"] - robustness["model"].map(base_change)
robustness["retention"] = robustness["recall@5%FPR_test"] / robustness["recall@5%FPR_val"]

robustness["rank_performance"] = robustness.groupby("dataset")["recall@5%FPR_test"].rank(ascending=False)
# shift_effect is 0 by construction on Base, so rank Base on its raw change instead
robustness["stability"] = np.where(robustness["dataset"] == "Base",
                                   robustness["recall@5%FPR_change"], robustness["shift_effect"])
robustness["rank_stability"] = robustness.groupby("dataset")["stability"].rank(ascending=False)
robustness = robustness.drop(columns="stability")

display(robustness.round(3))
robustness.to_csv(OUT_DIR + "/robustness.csv", index=False)

### 7.2 Precision–recall curves and best model
The "best" model for the detailed plots below is the one with the highest **Base validation AP**, so it is chosen without looking at test data.

In [ ]:
# pick the model on Base validation only, so the test months stay untouched
base_val_rank = rank_period[(rank_period["dataset"] == "Base") & (rank_period["split"] == "val")]
BEST = base_val_rank.sort_values("AP", ascending=False)["model"].iloc[0]
print("Best model on Base validation AP:", BEST)

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5), sharey=True)
for ax, dataset in zip(axes, datasets):
    for model_name in MODEL_NAMES:
        for split, style in [("val", "--"), ("test", "-")]:
            split_preds = preds[(preds["dataset"] == dataset) & (preds["model"] == model_name) & (preds["split"] == split)]
            precision, recall, _ = precision_recall_curve(split_preds["y"], split_preds["score"])
            width = 1.5 if model_name == BEST else 0.8
            label = model_name + " (" + split + ")" if dataset == "Base" else None
            ax.plot(recall, precision, style, lw=width, label=label)
    ax.set_title(dataset + " (dashed = val, solid = test)")
    ax.set_xlabel("recall")
    ax.set_ylabel("precision")
    ax.set_ylim(0, 1)
axes[0].legend(fontsize=7)
plt.tight_layout()
fig.savefig(OUT_DIR + "/fig5_pr_curves.png", dpi=200, bbox_inches="tight")
plt.show()

## 8. Operating setting A — fixed threshold (primary)
The threshold giving 5% FPR on the validation months is applied unchanged to each test month. Because the threshold is fixed, the alert rate (share of applications sent for review) can drift.

In [ ]:
val_preds = preds[preds["split"] == "val"]
threshold_rows = []
for (dataset, model_name), model_val in val_preds.groupby(["dataset", "model"]):
    threshold = np.quantile(model_val[model_val["y"] == 0]["score_tb"], 1 - TARGET_FPR)
    threshold_rows.append({"dataset": dataset, "model": model_name, "thr": threshold})
thresholds = pd.DataFrame(threshold_rows)

# drop first so re-running the cell doesn't produce thr_x / thr_y
preds = preds.drop(columns="thr", errors="ignore")
preds = preds.merge(thresholds, on=["dataset", "model"])
preds["flag_thr"] = preds["score_tb"] > preds["thr"]

thr_period = flag_stats(preds, "flag_thr", ["dataset", "model", "split"])
thr_month = flag_stats(preds, "flag_thr", ["dataset", "model", "month"])
display(thr_period.pivot_table(index=["dataset", "model"], columns="split",
                               values=["recall", "FPR", "precision", "alert_rate"]))
thr_period.to_csv(OUT_DIR + "/fixed_threshold_period.csv", index=False)
thr_month.to_csv(OUT_DIR + "/fixed_threshold_month.csv", index=False)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 7), sharex=True)
for j, dataset in enumerate(datasets):
    for i, metric in enumerate(["alert_rate", "recall"]):
        ax = axes[i, j]
        for model_name in MODEL_NAMES:
            model_month = thr_month[(thr_month["dataset"] == dataset) & (thr_month["model"] == model_name)]
            ax.plot(model_month["month"], model_month[metric], "-o", label=model_name)
        ax.axvspan(TEST_MONTHS[0] - 0.5, TEST_MONTHS[-1] + 0.5, color="grey", alpha=0.15)
        ax.set_title(dataset + " - " + metric + " (fixed threshold)")
        ax.set_ylabel(metric)
axes[0, 0].legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, dataset in zip(axes, datasets):
    test_preds = preds[(preds["dataset"] == dataset) & (preds["model"] == BEST) & (preds["split"] == "test")]
    cm = confusion_matrix(test_preds["y"], test_preds["flag_thr"])
    ConfusionMatrixDisplay(cm, display_labels=["legit", "fraud"]).plot(ax=ax, colorbar=False, values_format=",")
    ax.set_title(dataset + " - " + BEST + "\n(test, fixed threshold)")
plt.tight_layout()
fig.savefig(OUT_DIR + "/fig6_confusion_matrices.png", dpi=200, bbox_inches="tight")
plt.show()

## 9. Operating setting B — fixed review capacity (secondary)
Each month, the highest-scoring 1% and 5% of applications are sent for review. Workload is constant and the threshold changes month to month. Only that month's scores are used (no labels), so there is no leakage.

In [ ]:
# rank within each month so the review workload stays fixed even if the scores drift
month_scores = preds.groupby(["dataset", "model", "month"])["score_tb"]
month_size = month_scores.transform("size")
rank_in_month = month_scores.rank(method="first", ascending=False)

cap_tables = []
for cap in REVIEW_CAPACITIES:
    col = "flag_cap" + str(int(cap * 100))
    preds[col] = rank_in_month <= np.ceil(cap * month_size)
    cap_stats = flag_stats(preds, col, ["dataset", "model", "split"])
    cap_stats["capacity"] = cap
    cap_tables.append(cap_stats)
cap_period = pd.concat(cap_tables, ignore_index=True)

display(cap_period.pivot_table(index=["dataset", "model"], columns=["capacity", "split"], values=["recall", "FPR"]))
cap_period.to_csv(OUT_DIR + "/fixed_capacity_period.csv", index=False)

## 10. Age-group comparison (≤50 vs >50)
**Primary analysis: fixed threshold.** This isolates how each model's scores behave for each group. The fixed-capacity results are a secondary check: under fixed capacity, if one group's scores rise, that group takes up more review slots, and both groups' rates shift as a result.

`FPR_ratio_>50/<=50` > 1 means older applicants are wrongly flagged more often.

In [ ]:
age_gaps = {}
for flag_col in ["flag_thr"] + ["flag_cap" + str(int(cap * 100)) for cap in REVIEW_CAPACITIES]:
    age_stats = flag_stats(preds, flag_col, ["dataset", "model", "split", "age_group"])
    if flag_col == "flag_thr":
        age_thr = age_stats

    gap = age_stats.pivot_table(index=["dataset", "model", "split"], columns="age_group", values=["FPR", "recall"])
    gap.columns = [metric + "_" + group for metric, group in gap.columns]
    gap["FPR_ratio_>50/<=50"] = gap["FPR_>50"] / gap["FPR_<=50"]
    gap["recall_diff_>50-<=50"] = gap["recall_>50"] - gap["recall_<=50"]
    age_gaps[flag_col] = gap

gap_thr = age_gaps["flag_thr"]
print("PRIMARY - fixed threshold (5% FPR on validation)")
display(gap_thr)
for cap in REVIEW_CAPACITIES:
    print("\nSECONDARY - fixed capacity", cap)
    display(age_gaps["flag_cap" + str(int(cap * 100))])

gap_thr.to_csv(OUT_DIR + "/age_gap_fixed_threshold.csv")
gap_cap = {cap: age_gaps["flag_cap" + str(int(cap * 100))] for cap in REVIEW_CAPACITIES}
pd.concat(gap_cap, names=["capacity"]).to_csv(OUT_DIR + "/age_gap_fixed_capacity.csv")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 8), sharey="row")
for j, dataset in enumerate(datasets):
    for i, metric in enumerate(["FPR", "recall"]):
        ax = axes[i, j]
        by_age = age_thr[age_thr["dataset"] == dataset].pivot_table(index="model", columns=["split", "age_group"],
                                                                     values=metric)
        by_age = by_age[[("val", "<=50"), ("val", ">50"), ("test", "<=50"), ("test", ">50")]]
        by_age.plot.bar(ax=ax, rot=30, legend=(i == 0 and j == 0))
        ax.set_title(dataset + " - " + metric + " by age group (fixed threshold)")
        ax.set_xlabel("")
plt.tight_layout()
fig.savefig(OUT_DIR + "/fig3_age_groups.png", dpi=200, bbox_inches="tight")
plt.show()

### 10.1 Separability within each age group
Recall at 5% FPR recomputed **within each age group and period**, so it does not depend on the validation threshold. If a group's fixed-threshold recall falls but this stays steady, the threshold has gone stale for that group; if both fall, the model ranks that group's frauds worse.

In [ ]:
rank_age = preds.groupby(["dataset", "model", "split", "age_group"]).apply(rank_scores).reset_index()
display(rank_age.pivot_table(index=["dataset", "model"], columns=["age_group", "split"], values="recall@5%FPR").round(3))
rank_age.to_csv(OUT_DIR + "/ranking_by_age.csv", index=False)

### 10.2 Bootstrap confidence intervals
Rows of each period are resampled with replacement (500 times). The **same** resample is applied to every model, so differences between models are paired. Intervals are 95% percentile intervals; a difference whose interval contains 0 is not decisive.
* (a) recall at 5% FPR on validation and test, and the change, per model;
* (b) pairwise test-recall differences between models, per dataset;
* (c) the age-group gaps for the selected model at the fixed threshold;
* (d) the drop caused by each shift beyond Base, and the ratio of the Variant V drop to the Variant IV drop (RQ2).

In [ ]:
N_BOOT = 500
rng = np.random.default_rng(RANDOM_STATE)

# (a) per-model recall. every model is scored on the same resample, so the differences in (b) are paired
boot_rows = []
for (dataset, split), period_preds in preds[preds["split"].isin(["val", "test"])].groupby(["dataset", "split"]):
    scores_by_model = period_preds.pivot(index="row", columns="model", values="score_tb")
    y = period_preds.drop_duplicates("row").set_index("row")["y"].reindex(scores_by_model.index).values
    for b in range(N_BOOT):
        idx = rng.integers(0, len(y), len(y))
        for model_name in scores_by_model.columns:
            boot_rows.append({"dataset": dataset, "split": split, "model": model_name, "b": b,
                              "recall": recall_at_fpr(y[idx], scores_by_model[model_name].values[idx])})

boot = pd.DataFrame(boot_rows).pivot_table(index=["dataset", "model", "b"], columns="split", values="recall")
boot = boot.reset_index()
boot["change"] = boot["test"] - boot["val"]

by_model = boot.groupby(["dataset", "model"])
ci_model = pd.DataFrame({
    "test": by_model["test"].mean(),
    "test_lo": by_model["test"].quantile(0.025),
    "test_hi": by_model["test"].quantile(0.975),
    "change": by_model["change"].mean(),
    "change_lo": by_model["change"].quantile(0.025),
    "change_hi": by_model["change"].quantile(0.975),
})
print("(a) Recall at 5% FPR on test and the val -> test change, with 95% CI")
display(ci_model.round(3))

# (b)
pair_rows = []
for dataset in datasets:
    test_recall = boot[boot["dataset"] == dataset].pivot(index="b", columns="model", values="test")
    for model_a, model_b in combinations(test_recall.columns, 2):
        diff = test_recall[model_a] - test_recall[model_b]
        lo, hi = diff.quantile(0.025), diff.quantile(0.975)
        pair_rows.append({"dataset": dataset, "model_a": model_a, "model_b": model_b,
                          "diff_a_minus_b": diff.mean(), "lo": lo, "hi": hi, "decisive": not (lo <= 0 <= hi)})
ci_pairs = pd.DataFrame(pair_rows)
print("(b) Pairwise differences in test recall (decisive = CI doesn't include 0)")
display(ci_pairs.round(3))

# (c)
age_rows = []
best_preds = preds[(preds["model"] == BEST) & preds["split"].isin(["val", "test"])]
for (dataset, split), period_preds in best_preds.groupby(["dataset", "split"]):
    y = period_preds["y"].values
    flagged = period_preds["flag_thr"].values
    over_50 = (period_preds["age_group"] == ">50").values
    for b in range(N_BOOT):
        idx = rng.integers(0, len(y), len(y))
        y_s, flagged_s, over_50_s = y[idx], flagged[idx], over_50[idx]
        fpr_over = flagged_s[over_50_s & (y_s == 0)].mean()
        fpr_under = flagged_s[~over_50_s & (y_s == 0)].mean()
        recall_over = flagged_s[over_50_s & (y_s == 1)].mean()
        recall_under = flagged_s[~over_50_s & (y_s == 1)].mean()
        age_rows.append({"dataset": dataset, "split": split,
                         "FPR_ratio_>50/<=50": fpr_over / fpr_under,
                         "recall_diff_>50-<=50": recall_over - recall_under})

age_boot = pd.DataFrame(age_rows)
by_period = age_boot.groupby(["dataset", "split"])
ci_age = pd.DataFrame()
for col in ["FPR_ratio_>50/<=50", "recall_diff_>50-<=50"]:
    ci_age[col + "_mean"] = by_period[col].mean()
    ci_age[col + "_lo"] = by_period[col].quantile(0.025)
    ci_age[col + "_hi"] = by_period[col].quantile(0.975)
print("(c) Age group gaps for", BEST, "(fixed threshold)")
display(ci_age.round(3))

# (d) RQ2: drop caused by each shift beyond Base, and how many times worse Variant V is than Variant IV.
# each dataset was resampled independently, so draw b of one can be combined with draw b of another
changes = boot.pivot_table(index=["model", "b"], columns="dataset", values="change").reset_index()
models_avg = changes.groupby("b")[list(datasets)].mean().reset_index()
models_avg["model"] = "All models (mean)"
changes = pd.concat([changes, models_avg], ignore_index=True)
changes["IV_shift"] = changes["Variant IV"] - changes["Base"]
changes["V_shift"] = changes["Variant V"] - changes["Base"]
changes["V_vs_IV_ratio"] = changes["V_shift"] / changes["IV_shift"]

by_shift_model = changes.groupby("model")
ci_shift = pd.DataFrame()
for col in ["IV_shift", "V_shift", "V_vs_IV_ratio"]:
    ci_shift[col] = by_shift_model[col].mean()
    ci_shift[col + "_lo"] = by_shift_model[col].quantile(0.025)
    ci_shift[col + "_hi"] = by_shift_model[col].quantile(0.975)
print("(d) Recall drop caused by each shift (beyond Base) and the Variant V / Variant IV ratio")
display(ci_shift.round(3))

ci_model.to_csv(OUT_DIR + "/ci_recall_by_model.csv")
ci_shift.to_csv(OUT_DIR + "/ci_shift_effects.csv")
ci_pairs.to_csv(OUT_DIR + "/ci_model_differences.csv", index=False)
ci_age.to_csv(OUT_DIR + "/ci_age_gaps.csv")

### 10.3 Training without customer_age
Is the >50 vs ≤50 gap caused by the model using age directly? Logistic regression and the selected model are retrained on every dataset without `customer_age` (same hyperparameters), the 5% FPR validation threshold is re-derived, and the age-group FPR ratio and recall are compared with the original models. If the ratio stays near 3, other features that correlate with age carry the disparity.

In [ ]:
focus_models = ["Logistic Regression"]
if BEST != "Logistic Regression":
    focus_models.append(BEST)

noage_frames = []
for dataset, df in datasets.items():
    train = df[df["split"] == "train"].drop(columns=[AGE])
    eval_df = df[df["split"] != "train"]
    for model_name in focus_models:
        pipe, features = train_model(train, model_name, best_params[model_name])
        noage_frames.append(pd.DataFrame({
            "dataset": dataset,
            "model": model_name,
            "row": eval_df.index,
            "month": eval_df[MONTH].values,
            "split": eval_df["split"].values,
            "y": eval_df[TARGET].values,
            "score": pipe.predict_proba(eval_df[features])[:, 1],
            "age_group": eval_df["age_group"].values,
        }))
        print(dataset, "-", model_name, "without customer_age done")
noage = pd.concat(noage_frames, ignore_index=True)
noage["score_tb"] = noage["score"] + np.random.default_rng(RANDOM_STATE).uniform(0, 1e-9, len(noage))
noage["features"] = "no customer_age"

with_age = preds[preds["model"].isin(focus_models)][list(noage.columns.drop("features"))].copy()
with_age["features"] = "all"
age_compare = pd.concat([with_age, noage], ignore_index=True)

# each version gets its own 5% FPR threshold from validation, as in section 8
val_legit = age_compare[(age_compare["split"] == "val") & (age_compare["y"] == 0)]
thr_rows = []
for (features, dataset, model_name), group in val_legit.groupby(["features", "dataset", "model"]):
    thr_rows.append({"features": features, "dataset": dataset, "model": model_name,
                     "thr": np.quantile(group["score_tb"], 1 - TARGET_FPR)})
age_compare = age_compare.merge(pd.DataFrame(thr_rows), on=["features", "dataset", "model"])
age_compare["flag_thr"] = age_compare["score_tb"] > age_compare["thr"]

keys = ["features", "dataset", "model", "split"]
noage_gap = flag_stats(age_compare, "flag_thr", keys + ["age_group"]).pivot_table(
    index=keys, columns="age_group", values=["FPR", "recall"])
noage_gap.columns = [metric + "_" + group for metric, group in noage_gap.columns]
noage_gap["FPR_ratio_>50/<=50"] = noage_gap["FPR_>50"] / noage_gap["FPR_<=50"]
noage_gap["recall_diff_>50-<=50"] = noage_gap["recall_>50"] - noage_gap["recall_<=50"]
overall = age_compare.groupby(keys).apply(rank_scores)
noage_gap["AP"] = overall["AP"]
noage_gap["recall@5%FPR"] = overall["recall@5%FPR"]

display(noage_gap.round(3))
noage_gap.to_csv(OUT_DIR + "/fairness_without_age.csv")

### 10.4 Separate thresholds per age group
Instead of one threshold, each age group gets its own threshold giving 5% FPR on that group's validation applications (no retraining). This equalises FPR on validation by construction; the question is how much recall it costs, and whether the equality still holds on the test months after the shift.

In [ ]:
val_legit = preds[(preds["split"] == "val") & (preds["y"] == 0)]
group_thr_rows = []
for (dataset, model_name, age_group), group in val_legit.groupby(["dataset", "model", "age_group"]):
    group_thr_rows.append({"dataset": dataset, "model": model_name, "age_group": age_group,
                           "group_thr": np.quantile(group["score_tb"], 1 - TARGET_FPR)})
# merge into a copy: changing preds' row order would change the bootstrap draws above on a re-run
group_preds = preds.merge(pd.DataFrame(group_thr_rows), on=["dataset", "model", "age_group"])
group_preds["flag_group_thr"] = group_preds["score_tb"] > group_preds["group_thr"]

rule_tables = []
for rule, flag_col, rule_preds in [("single threshold", "flag_thr", preds),
                                   ("per-group threshold", "flag_group_thr", group_preds)]:
    by_age = flag_stats(rule_preds, flag_col, ["dataset", "model", "split", "age_group"])
    gap = by_age.pivot_table(index=["dataset", "model", "split"], columns="age_group", values=["FPR", "recall"])
    gap.columns = [metric + "_" + group for metric, group in gap.columns]
    gap["FPR_ratio_>50/<=50"] = gap["FPR_>50"] / gap["FPR_<=50"]
    gap["recall_diff_>50-<=50"] = gap["recall_>50"] - gap["recall_<=50"]

    overall = flag_stats(rule_preds, flag_col, ["dataset", "model", "split"]).set_index(["dataset", "model", "split"])
    gap["recall"] = overall["recall"]
    gap["FPR"] = overall["FPR"]
    gap["alert_rate"] = overall["alert_rate"]
    gap["rule"] = rule
    rule_tables.append(gap.reset_index())
group_thr_results = pd.concat(rule_tables, ignore_index=True)

display(group_thr_results.pivot_table(index=["dataset", "model", "rule"], columns="split",
                                      values=["FPR_ratio_>50/<=50", "recall"]).round(3))
group_thr_results.to_csv(OUT_DIR + "/group_thresholds.csv", index=False)

## 11. Interpretation
### 11.1 Stability of feature predictiveness over time
For each numeric feature, univariate predictive strength = |2·AUC − 1| (0 = no signal, 1 = perfect separation), computed per month and age group. Variant V should show signals that fade in months 6–7 for one age group.

In [ ]:
strength_rows = []
for dataset, df in datasets.items():
    train = df[df["split"] == "train"]
    # use each dataset's own columns, otherwise Variant V's x1/x2 get left out
    num_features = [c for c in df.columns
                    if c not in CAT_COLS + [TARGET, MONTH, "age_group", "split"]
                    and train[c].nunique(dropna=False) > 1]
    for (month, age_group), month_group in df.groupby([MONTH, "age_group"]):
        if month_group[TARGET].nunique() < 2:
            continue
        for col in num_features:
            values = month_group[col].fillna(month_group[col].median())
            auc = roc_auc_score(month_group[TARGET], values)
            strength_rows.append({"month": month, "age_group": age_group, "feature": col,
                                  "strength": abs(2 * auc - 1), "dataset": dataset})
strength = pd.DataFrame(strength_rows)
strength.to_csv(OUT_DIR + "/feature_strength.csv", index=False)

base_train_strength = strength[(strength["dataset"] == "Base") & (strength["month"].isin(TRAIN_MONTHS))]
top_feats = list(base_train_strength.groupby("feature")["strength"].mean().nlargest(10).index)
# the shifts are injected through these, so always show them even if they're not top 10 on Base
for col in ["customer_age", "x1", "x2"]:
    if col not in top_feats:
        top_feats.append(col)

fig, axes = plt.subplots(3, 2, figsize=(15, 13))
for i, dataset in enumerate(datasets):
    for j, age_group in enumerate(["<=50", ">50"]):
        group_strength = strength[(strength["dataset"] == dataset) & (strength["age_group"] == age_group)
                                  & (strength["feature"].isin(top_feats))]
        heat = (group_strength.pivot(index="feature", columns="month", values="strength")
                .reindex(top_feats).dropna(how="all"))
        im = axes[i, j].imshow(heat.values, aspect="auto", cmap="viridis", vmin=0, vmax=strength["strength"].max())
        axes[i, j].set_title(dataset + " - age " + age_group)
        axes[i, j].set_xticks(range(heat.shape[1]))
        axes[i, j].set_xticklabels(heat.columns)
        axes[i, j].set_yticks(range(len(heat)))
        axes[i, j].set_yticklabels(heat.index)
        axes[i, j].set_xlabel("month")
fig.colorbar(im, ax=axes, shrink=0.6, label="|2*AUC - 1|")
fig.savefig(OUT_DIR + "/fig4_feature_strength.png", dpi=200, bbox_inches="tight")
plt.show()

shift_feats = strength[strength["feature"].isin(["customer_age", "x1", "x2"])].copy()
shift_feats["period"] = np.where(shift_feats["month"].isin(TEST_MONTHS), "months 6-7", "months 0-5")
display(shift_feats.pivot_table(index=["dataset", "feature", "age_group"], columns="period", values="strength").round(3))

### 11.1b Feature distribution drift (PSI)
Section 11.1 measures how *predictive* each feature is. This checks whether the feature *values* themselves move between the training months (0–3) and the test months (6–7), using the Population Stability Index: < 0.1 stable, 0.1–0.25 moderate, > 0.25 large drift. Low PSI alongside a fall in predictive strength means the shift is in feature predictiveness, not feature distributions.

In [ ]:
drift_rows = []
for dataset, df in datasets.items():
    train = df[df["split"] == "train"]
    test = df[df["split"] == "test"]
    features = [c for c in df.columns if c not in [TARGET, MONTH, "age_group", "split"]
                and train[c].nunique(dropna=False) > 1]
    for col in features:
        train_vals = train[col].dropna()
        test_vals = test[col].dropna()

        edges = []
        if col not in CAT_COLS:
            edges = np.unique(np.quantile(train_vals, np.linspace(0, 1, 11)))

        # binary / low-cardinality columns collapse to < 3 quantile edges, so compare value shares instead
        if len(edges) >= 3:
            expected = np.histogram(train_vals, edges)[0] / len(train_vals)
            actual = np.histogram(test_vals.clip(edges[0], edges[-1]), edges)[0] / len(test_vals)
        else:
            categories = np.union1d(train_vals.unique(), test_vals.unique())
            expected = train_vals.value_counts(normalize=True).reindex(categories, fill_value=0).values
            actual = test_vals.value_counts(normalize=True).reindex(categories, fill_value=0).values

        # avoid log(0) for bins that are empty in one period
        expected = np.clip(expected, 1e-6, None)
        actual = np.clip(actual, 1e-6, None)
        psi = np.sum((actual - expected) * np.log(actual / expected))
        drift_rows.append({"feature": col, "dataset": dataset, "psi": psi})

drift = pd.DataFrame(drift_rows).pivot(index="feature", columns="dataset", values="psi")
display(drift.sort_values("Base", ascending=False).round(3))
for dataset in datasets:
    print(dataset, "- features with PSI > 0.1:", list(drift.index[drift[dataset] > 0.1]))
drift.to_csv(OUT_DIR + "/feature_drift_psi.csv")

### 11.2 Permutation importance (selected model): validation vs test
Importance is measured as the drop in AP when a feature is shuffled, on a sample of each period. Features that matter on validation but not on test indicate reliance on unstable relationships.

In [ ]:
PI_SAMPLE = 50000   # permutation importance on the full 250k rows per period is too slow
pi_rows = []
for dataset, df in datasets.items():
    pipe, features = fitted[(dataset, BEST)]
    for split in ["val", "test"]:
        period_df = df[df["split"] == split]
        period_df = period_df.sample(min(PI_SAMPLE, len(period_df)), random_state=RANDOM_STATE)
        perm = permutation_importance(pipe, period_df[features], period_df[TARGET], scoring="average_precision",
                                      n_repeats=3, random_state=RANDOM_STATE, n_jobs=1)
        for feature, importance in zip(features, perm.importances_mean):
            pi_rows.append({"dataset": dataset, "split": split, "feature": feature, "importance": importance})

pi = pd.DataFrame(pi_rows).pivot_table(index=["dataset", "feature"], columns="split", values="importance")
pi["change"] = pi["test"] - pi["val"]
for dataset in datasets:
    print("\n" + dataset + " - top 10 features by validation importance (" + BEST + ")")
    display(pi.loc[dataset].sort_values("val", ascending=False).head(10))
pi.to_csv(OUT_DIR + "/permutation_importance.csv")

### 11.3 Error analysis: frauds missed in the test period
Using the selected model at the fixed threshold, test-period frauds are split into caught vs missed, by age group, and their feature profiles are compared.

In [ ]:
error_by_age = []
error_profiles = []
for dataset, df in datasets.items():
    test_frauds = preds[(preds["dataset"] == dataset) & (preds["model"] == BEST)
                        & (preds["split"] == "test") & (preds["y"] == 1)]
    print("\n=== " + dataset + " ===")
    caught = test_frauds.groupby("age_group")["flag_thr"].agg(frauds="size", caught="sum", recall="mean")
    display(caught)
    caught.insert(0, "dataset", dataset)
    error_by_age.append(caught.reset_index())

    fraud_profiles = df.loc[test_frauds["row"]].copy()
    fraud_profiles["outcome"] = np.where(test_frauds["flag_thr"].values, "caught", "missed")
    profile_cols = [c for c in top_feats if c in df.columns]
    profile = fraud_profiles.groupby(["age_group", "outcome"])[profile_cols].median()
    display(profile.T)
    profile.insert(0, "dataset", dataset)
    error_profiles.append(profile.reset_index())

pd.concat(error_by_age, ignore_index=True).to_csv(OUT_DIR + "/error_analysis_by_age.csv", index=False)
pd.concat(error_profiles, ignore_index=True).to_csv(OUT_DIR + "/error_analysis_profiles.csv", index=False)

### 11.4 Ablation: remove the shifted features and retrain
A direct test of RQ2 within each variant (no comparison with Base needed). Variant IV is retrained without `customer_age`, Variant V without `x1` and `x2`, keeping the same hyperparameters. If the validation-to-test drop mostly disappears, that feature is what carried the shift. Results are split by age group for RQ3.

In [ ]:
ablations = {"Variant IV": ["customer_age"], "Variant V": ["x1", "x2"]}
abl_models = ["Logistic Regression"]
if BEST != "Logistic Regression":
    abl_models.append(BEST)

abl_frames = []
for dataset, drop_cols in ablations.items():
    df = datasets[dataset]
    train = df[df["split"] == "train"].drop(columns=drop_cols)
    eval_df = df[df["split"] != "train"]
    for model_name in abl_models:
        pipe, features = train_model(train, model_name, best_params[model_name])
        scored = pd.DataFrame({"y": eval_df[TARGET].values, "split": eval_df["split"].values,
                               "age_group": eval_df["age_group"].values})
        scored["score"] = pipe.predict_proba(eval_df[features])[:, 1]
        scored["score_tb"] = scored["score"]   # LR and XGBoost scores are continuous, no ties to break

        overall = scored.groupby("split").apply(rank_scores).reset_index()
        overall["age_group"] = "all"
        per_age = scored.groupby(["split", "age_group"]).apply(rank_scores).reset_index()

        abl_metrics = pd.concat([overall, per_age])
        abl_metrics["dataset"] = dataset
        abl_metrics["model"] = model_name
        abl_metrics["dropped"] = "+".join(drop_cols)
        abl_frames.append(abl_metrics)
        print(dataset, "-", model_name, "without", drop_cols, "done")
abl = pd.concat(abl_frames, ignore_index=True)
abl["features"] = "shift features removed"

full_preds = preds[preds["dataset"].isin(list(ablations)) & preds["model"].isin(abl_models)]
full_overall = full_preds.groupby(["dataset", "model", "split"]).apply(rank_scores).reset_index()
full_overall["age_group"] = "all"
full_per_age = full_preds.groupby(["dataset", "model", "split", "age_group"]).apply(rank_scores).reset_index()
full_metrics = pd.concat([full_overall, full_per_age])
full_metrics["features"] = "all"

ablation_results = pd.concat([full_metrics, abl], ignore_index=True)
recall_table = ablation_results.pivot_table(index=["dataset", "model", "age_group", "features"],
                                            columns="split", values="recall@5%FPR")
recall_table["change"] = recall_table["test"] - recall_table["val"]
display(recall_table.round(3))
ablation_results.to_csv(OUT_DIR + "/ablation.csv", index=False)

### 11.5 Retraining on more recent months
Would a bank that retrains more often have avoided the loss? Logistic regression and the selected model are retrained on months 0–5 (all labelled history) and on months 2–5 (recent only), with the same hyperparameters, and scored on the same test months 6–7. Ranking metrics are used because no validation months are left to set a threshold. In Variant V the shift starts in month 6, so retraining can only help if the shift is already visible in the training labels.

In [ ]:
TRAIN_WINDOWS = {"months 0-5": [0, 1, 2, 3, 4, 5], "months 2-5": [2, 3, 4, 5]}

retrain_rows = []
# original models (months 0-3) from section 5, for comparison
orig_test = preds[(preds["split"] == "test") & preds["model"].isin(focus_models)]
for (dataset, model_name), model_test in orig_test.groupby(["dataset", "model"]):
    for age_group, group in [("all", model_test)] + list(model_test.groupby("age_group")):
        retrain_rows.append({"dataset": dataset, "model": model_name, "train_months": "months 0-3",
                             "age_group": age_group, **rank_scores(group).to_dict()})

for dataset, df in datasets.items():
    test = df[df["split"] == "test"]
    for window, months in TRAIN_WINDOWS.items():
        train = df[df[MONTH].isin(months)]
        for model_name in focus_models:
            pipe, features = train_model(train, model_name, best_params[model_name])
            scored = pd.DataFrame({"y": test[TARGET].values, "age_group": test["age_group"].values,
                                   "score": pipe.predict_proba(test[features])[:, 1]})
            scored["score_tb"] = scored["score"]
            for age_group, group in [("all", scored)] + list(scored.groupby("age_group")):
                retrain_rows.append({"dataset": dataset, "model": model_name, "train_months": window,
                                     "age_group": age_group, **rank_scores(group).to_dict()})
            print(dataset, "-", model_name, "trained on", window, "done")
retraining = pd.DataFrame(retrain_rows)

display(retraining.pivot_table(index=["dataset", "model", "age_group"], columns="train_months",
                               values="recall@5%FPR").round(3))
retraining.to_csv(OUT_DIR + "/retraining.csv", index=False)

## 12. Summary table
One row per dataset × model with the headline numbers for the report.

In [ ]:
summary = robust[["AP_val", "AP_test", "AP_change", "recall@5%FPR_val", "recall@5%FPR_test", "recall@5%FPR_change"]].copy()

thr_test = thr_period[thr_period["split"] == "test"].set_index(["dataset", "model"])
summary["thr_test_recall"] = thr_test["recall"]
summary["thr_test_FPR"] = thr_test["FPR"]
summary["thr_test_alert_rate"] = thr_test["alert_rate"]

age_test = gap_thr.xs("test", level="split")
summary["age_test_FPR_ratio_>50/<=50"] = age_test["FPR_ratio_>50/<=50"]
summary["age_test_recall_diff_>50-<=50"] = age_test["recall_diff_>50-<=50"]

for cap in REVIEW_CAPACITIES:
    cap_test = cap_period[(cap_period["capacity"] == cap) & (cap_period["split"] == "test")].set_index(["dataset", "model"])
    summary["cap" + str(int(cap * 100)) + "_test_recall"] = cap_test["recall"]

display(summary)
summary.to_csv(OUT_DIR + "/summary.csv")

## Notes and limitations
* Scores are tie-broken with negligible random noise before thresholding (see section 5), so fixed-FPR and top-k rules behave sensibly for models with few distinct scores such as the decision tree.
* LightGBM runs in deterministic mode and both boosting libraries use a fixed thread count (`N_THREADS`), so re-runs on the same machine give identical results; different machines or library versions can still differ slightly. The imports cell prints the library versions used.
* The BAF data are synthetic; results are controlled evidence about model behaviour under imbalance and temporal shift, not production performance estimates.
* The model used for testing is the one trained on months 0–3, so the validation-chosen threshold matches its scores exactly. Refitting on months 0–5 would use more data but would require re-deriving the threshold (e.g. by cross-fitting).
* Hyperparameters were chosen by Optuna on a single time-respecting holdout (months 4–5 of Base); an expanding-window scheme is a possible extension.